### IMPORTAÇÕES

In [1]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd
import awswrangler as awr
import openpyxl
import shutil
import datetime as dt
import pyautogui
import time
import matplotlib.pyplot as plt
import matplotlib
import os
matplotlib.use('Agg')  

In [2]:
# FORMATANDO EM CAIXA ALTA
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

### EXTRAÇÕES DE BASES

In [3]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos =awr.athena.read_sql_query(
    sql=sql_ativos,database='silver'
)



In [4]:
# ELIMINANDO DUPLICATAS DE ATIVOS POR CHASSI

df_ativos = df_ativos.drop(columns=['rn'])

df_ativos = (
    df_ativos
    .drop_duplicates(subset=['chassi'], keep='first')
)

df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [5]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver'
)

In [6]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver'
)

In [7]:
# FORMATAÇÃO EM CAIXA ALTA
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,MLD9B48,9EP071120D1001811,9357,0,9357,17284,16550,UNIDADE PORTO VELHO,ELIZETE CHAGAS,ATIVO,...,,102290,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-10,2025-12-10,2025-12-30,2025-12-30,2026-01-26,STCOOP
1,MLD9B78,9EP070820D1001808,9356,0,9356,17284,16550,UNIDADE PORTO VELHO,ELIZETE CHAGAS,ATIVO,...,,102289,CASCO (R/SR),CANCELADO,2026-02-10,2025-12-10,2025-12-30,2025-12-30,2026-01-26,STCOOP
2,MLG3A25,9BVAG40DXDE808439,1709,1709,<NA>,18823,15737,UNIDADE CUIABA,ROSILDA CRUZ,ATIVO,...,,97536,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-02-10,2025-09-11,2025-09-21,2025-09-21,2026-01-21,STCOOP
3,ADU6J30,34542012290717,23790,23790,<NA>,20529,14019,UNIDADE MARINGÁ,HUGO SIGAKI,ATIVO,...,,87138,REPARAÇÃO A TERCEIROS,CANCELADO,2026-02-10,2025-04-01,2025-04-03,2025-04-03,2026-01-19,STCOOP
4,MDH6477,9BW2M82T04R435501,26679,26679,<NA>,21222,15344,UNIDADE CASCAVEL,MAURICIO BELLO,ATIVO,...,,95270,REPARAÇÃO A TERCEIROS,CANCELADO,2026-02-10,2025-07-31,2025-08-08,2025-08-08,2025-09-09,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2453,RRP9A90,94BL0262NNR068672,27596,0,27596,4293,5896,UNIDADE SINOP,QUEILA DE CAMARGO REIS DOS SANTOS,ATIVO,...,,740975,ASSISTÊNCIA A REPAROS (R/SR),CANCELADO,2026-02-10,2025-10-22,2025-10-23,2025-10-23,2025-12-30,TAG
2454,RRP9A80,94BB0902NNR068671,27598,0,27598,4293,5896,UNIDADE SINOP,QUEILA DE CAMARGO REIS DOS SANTOS,ATIVO,...,,740974,CASCO PT (R/SR),CANCELADO,2026-02-10,2025-10-22,2025-10-23,2025-10-23,2025-12-30,TAG
2455,RRP9B00,94BB0902NNR068670,27595,0,27595,4293,5896,UNIDADE SINOP,QUEILA DE CAMARGO REIS DOS SANTOS,ATIVO,...,,740974,CASCO PT (R/SR),CANCELADO,2026-02-10,2025-10-22,2025-10-23,2025-10-23,2025-12-30,TAG
2456,ABM5I17,9BM938142LS056920,1475383,1475383,<NA>,4490,6211,MICRO A - LUANA SOARES ALVES DOS REIS,LUANA SOARES ALVES DOS REIS,ATIVO,...,,744191,SUPORTE EMERGENCIAL MONITORADO,CANCELADO,2026-02-10,2025-10-28,2025-10-31,2025-11-04,2025-11-26,TAG


In [8]:
# DEFININDO TODAY TS e YESTERDAY

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')

yesterday

'09-02-2026'

### JUNÇÃO DE BASES DE CANCELAMENTO

In [9]:
# FORMATANDO COLUNAS DE DATA DE CANCELAMENTO PARA DT (TS)

df_cancelamentos_integrais['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos_integrais['data_cancelamento'], errors='coerce'
).dt.date

placas_canceladas_dia_anterior = df_cancelamentos_integrais.loc[
	df_cancelamentos_integrais['data_cancelamento'] == yesterday_ts.date(),
	'placa'
].unique().tolist()

In [10]:
# CRIANDO COLUNAS DE IDENTIFICADOR DE CANCELAMENTO

df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [11]:
# CONCATENANDO BASES DE CANCELAMENTO E RETIRANDO DUPLICATAS POR CHASSI

df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos.drop_duplicates(subset=['chassi'], keep='first')

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_cancelamento,empresa,identificador,data_atualizacao
0,MMB0A33,9BSTH4X2ZV3265532,17533,17533,<NA>,14260,13266,UNIDADE LONDRINA,DAIANE CRISTINA VEIGA DA SILVA,FINALIZADO,...,REPARAÇÃO A TERCEIROS,ATIVO,2026-02-10,2025-01-22,2025-01-25,2025-01-25,2026-02-10,STCOOP,INTEGRAL,NaT
1,AXP4I51,9EP021330E1000151,12043,0,12043,5711,8727,UNIDADE LONDRINA,DAIANE CRISTINA VEIGA DA SILVA,FINALIZADO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,ATIVO,2026-02-10,2025-01-22,2025-01-25,2025-01-25,2026-02-10,VIAVANTE,INTEGRAL,NaT
2,BEA0E92,9EP021330D1000993,12032,0,12032,5711,8727,UNIDADE LONDRINA,DAIANE CRISTINA VEIGA DA SILVA,FINALIZADO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-02-10,2025-01-22,2025-01-25,2025-01-25,2026-02-10,VIAVANTE,INTEGRAL,NaT
3,SDY9H92,9AA93113NPC203494,13787,0,13787,2981,8730,MF - MICRO FRANQUEADO J BATISTA VOLPATO REPRE...,JOÃO BATISTA VOLPATO,FINALIZADO,...,CASCO (R/SR),ATIVO,2026-02-10,2025-01-22,2025-01-24,2025-01-24,2026-02-09,VIAVANTE,INTEGRAL,NaT
4,MMK0H41,9BVAG20C3DE798891,15930223,15930223,<NA>,5746,8776,MICRO B - JULIANO DE COSTA,JULIANO COSTA,FINALIZADO,...,CASCO (VEÍCULO),ATIVO,2026-02-10,2025-01-23,2025-01-24,2025-01-24,2026-02-09,VIAVANTE,INTEGRAL,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39547,RAJ2014,96AF1483KLJ002933,326,0,326,2110,2603,P.I - T.V PASQUINI SERVICOS ADMINISTRATIVOS LTDA,TIAGO VINICIUS PASQUINI,ATIVO,...,RASTREADOR P/ REBOQUE,CANCELADO,2026-02-10,2025-09-03,2025-09-05,2025-09-05,NaN,TAG,PARCIAL,2026-01-14
39554,AZY1D90,9536T8270DR312360,261624,261624,<NA>,202,1278,FRANQUEADO - LEAO DE JUDA ADMINISTRACOES & PAR...,LEAO DE JUDA JEFFERSON ELYESER NEUMANN,ATIVO,...,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-02-10,2025-08-20,2025-08-21,2025-08-21,NaN,TAG,PARCIAL,2025-08-25
39573,SEG6A01,9BM963414RB351239,1475309,1475309,<NA>,4490,6211,MICRO A - LUANA SOARES ALVES DOS REIS,LUANA SOARES ALVES DOS REIS,ATIVO,...,APP,CANCELADO,2026-02-10,2025-10-28,2025-10-31,2025-11-04,NaN,TAG,PARCIAL,2025-11-26
39594,JOR7F73,9BVN4B5A01E675707,413162,413162,<NA>,3102,4153,UNIDADE CUIABA,ROSILDA MARTINS DA CRUZ,ATIVO,...,ASSISTÊNCIA A VIDROS,CANCELADO,2026-02-10,2025-09-25,2025-09-29,2025-10-01,NaN,TAG,PARCIAL,2025-12-10


In [13]:
# DEFININDO TODAY e YESTERDAY

today = pd.Timestamp.today().date()

yesterday = today - dt.timedelta(days=1)
sexta = yesterday - dt.timedelta(days=2)
sabado = yesterday - dt.timedelta(days=1)

### CONSTRUINDO EXCEL

In [14]:
# DEFININDO CÉLULAS RELATÓRIO w4
df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

if today.weekday() == 0: 
    ativos_mask = (df_ativos['inicio_vig'] >= sexta) & (df_ativos['inicio_vig'] <= yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] >= sexta) & (df_cancelamentos['data_cancelamento'] <= yesterday)
else:
    ativos_mask = (df_ativos['inicio_vig'] == yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] == yesterday)

ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & ativos_mask])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & ativos_mask])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & ativos_mask])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & ativos_mask])
total_ativados = len(df_ativos[ativos_mask])


cancelados_seg = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask])
cancelados_st = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask])
cancelados_viav = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask])
cancelados_tag = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask])
total_cancelados = len(df_cancelamentos[cancelamentos_mask])

In [15]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [16]:
# DEFININDO NOMES DAS ABAS E FAZENDO A LIMPEZA DE w1 e w2

template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']



clear_sheet(w1)
clear_sheet(w2)

In [17]:
# FORMATANDO LINHAS NULAS

numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')

W1 - ATIVOS

In [18]:

for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

W2 - CANCELADOS

In [19]:

for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

W3 - BASE

In [23]:
first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value.date()


In [24]:
first_empty_row

191

In [22]:
data_atual_planilha

datetime.date(2026, 2, 6)

In [25]:



"""
# Certifique-se de que 'yesterday' e 'data_atual_planilha' sejam ambos date
if isinstance(data_atual_planilha, pd.Timestamp):
    data_atual_planilha = data_atual_planilha.date()
elif hasattr(data_atual_planilha, 'date'):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, dt.datetime):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, str):
    try:
        data_atual_planilha = pd.to_datetime(data_atual_planilha).date()
    except Exception:
        pass


# Também garanta que 'yesterday' é date
if isinstance(yesterday, pd.Timestamp):
    yesterday_date = yesterday.date()
elif isinstance(yesterday, dt.datetime):
    yesterday_date = yesterday.date()
else:
    yesterday_date = yesterday
"""


# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM  
#if data_atual_planilha == yesterday:
    # Filtra ativos até a data de ontem (inclusive)
if "inicio_vig" in df_ativos.columns:
    # Supondo que data_ativacao já seja datetime ou string convertível
    data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')
    df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
    qtd_ativos = df_ativos_filtrados['chassi'].nunique()
else:
    qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos

#definindo ativados totais

data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sabado)]
qtd_ativos_sab = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sexta)]
qtd_ativos_sex = df_ativos_filtrados['chassi'].nunique()


#definindo ativações de sexta e sábado
ativos_mask_sab = df_ativos[df_ativos['inicio_vig'] == sabado]
total_ativados_sab = len(df_ativos[ativos_mask])

ativos_mask_sex = df_ativos[df_ativos['inicio_vig'] == sexta]
total_ativados_sex= len(df_ativos[ativos_mask])

#definindo cancelamentos de sexta e sábado

cancel_mask_sab = df_cancelamentos['data_cancelamento'] == sabado
total_cancel_sab = len(df_cancelamentos[cancel_mask_sab])

cancel_mask_sex = df_cancelamentos['data_cancelamento'] == sexta
total_cancel_sex = len(df_cancelamentos[cancel_mask_sex])

hoje = dt.date.today()
if hoje.weekday() == 0 and first_empty_row >= 3:
        w3['B' + str(first_empty_row + 2)] = qtd_ativos
        w3['C' + str(first_empty_row + 2)] = total_ativados
        w3['D' + str(first_empty_row + 2)] = total_ativados

        w3['B' + str(first_empty_row + 1)] = qtd_ativos_sab
        w3['C' + str(first_empty_row + 1)] = total_ativados_sab
        w3['D' + str(first_empty_row + 1)] = total_cancel_sab

        w3['B' + str(first_empty_row)] = qtd_ativos_sex
        w3['C' + str(first_empty_row)] = total_ativados_sex
        w3['D' + str(first_empty_row)] = total_cancel_sex

        print('Registro de ativos preenchido para domingo, sábado e ativos na aba BASE!')
else:
        w3['B' + str(first_empty_row)] = qtd_ativos
        w3['C' + str(first_empty_row)] = total_ativados
        w3['D' + str(first_empty_row)] = total_cancelados
        



W4 - RESUMO

In [26]:
import datetime 
# Garante que yesterday esteja normalizado
if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

# Hoje
hoje = datetime.date.today()
dia_semana = hoje.weekday() 

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    sexta_str = sexta.strftime('%d/%m/%Y')
    domingo_str = domingo.strftime('%d/%m/%Y')
    resumo_periodo = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    w4['C2'] = resumo_periodo
else:
    w4['C2'] = yest_date.strftime('%d/%m/%Y')

w4['C3'] = qtd_ativos

w4['C6'] = ativados_seg
w4['C7'] = ativados_st
w4['C8'] = ativados_viav
w4['C9'] = ativados_tag

w4['D6'] = cancelados_seg
w4['D7'] = cancelados_st
w4['D8'] = cancelados_viav
w4['D9'] = cancelados_tag


### QUANTIDADE DE CHASSIS POR UNIDADE

In [27]:

# Obtém o dia da semana referente a 'yesterday'
if hasattr(yesterday, "weekday"):
    dia_semana = yesterday.weekday()
else:
    try:
        dia_semana = datetime.datetime.strptime(str(yesterday), "%Y-%m-%d").weekday()
    except Exception:
        dia_semana = None  # fallback

if dia_semana == 0:  # Segunda-feira
    # sexta = ontem - 2 dias
    sexta = yesterday - datetime.timedelta(days=2)
    # Ativos de sexta, sábado e domingo (considerando 'inicio_vig')
    period_mask = df_ativos["inicio_vig"].isin([sexta, sexta + datetime.timedelta(days=1), yesterday])
    df_ativos_periodo = df_ativos[period_mask]
    df_ativos_ontem_unidades = df_ativos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Ativos por unidade (sexta a domingo):")
    print(df_ativos_ontem_unidades)
    
    # Cancelamentos de sexta, sábado e domingo (considerando 'data_cancelamento')
    period_mask_cancel = df_cancelamentos["data_cancelamento"].isin([sexta, sexta + datetime.timedelta(days=1), yesterday])
    df_cancelamentos_periodo = df_cancelamentos[period_mask_cancel]
    df_cancelamentos_ontem_unidades = df_cancelamentos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Cancelamentos por unidade (sexta a domingo):")
    print(df_cancelamentos_ontem_unidades)

else:
    # Comportamento padrão: apenas o dia anterior
    df_ativos_ontem = df_ativos[df_ativos['inicio_vig'] == yesterday]
    df_ativos_ontem_unidades = df_ativos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Ativos por unidade (ontem):")
    print(df_ativos_ontem_unidades)

    df_cancelamentos_ontem = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]
    df_cancelamentos_ontem_unidades = df_cancelamentos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)
    print("Cancelamentos por unidade (ontem):")
    print(df_cancelamentos_ontem_unidades)



Ativos por unidade (sexta a domingo):
unidade
FRANQUEADO - R M DE OLIVEIRA CORRETORES                  15
FRANQUEADO - EMPRESAUTO ASSESSORIA EMPRESARIAL LTDA       8
UNIDADE CUIABA                                            5
UNIDADE ITAPEJARA                                         4
MICRO A - MICRO SIDINES BERTOLDI                          4
UNIDADE LUCAS DO RIO VERDE                                4
UNIDADE MARINGÁ                                           4
ASSESSORIA - PELTA GOIANIA                                3
ASSESSORIA - PELTA SÃO BERNARDO DO CAMPO                  2
UNIDADE CASCAVEL                                          2
UNIDADE CURITIBA                                          2
P.I - BRAVII SEGUROS E RASTREAMENTOS                      2
UNIDADE CARAZINHO                                         2
CP - EMGN REPRESENTACOES E TRANSPORTES DE CARGAS LTDA     1
MICRO A - MARTUCCI SOLUCOES LTDA                          1
TS CONSULTORIA EM TRANSPORTES LTDA                    

### criando imagem das unidades

In [28]:

# Garante colunas date
if not pd.api.types.is_datetime64_any_dtype(df_ativos['inicio_vig']):
    df_ativos['inicio_vig'] = pd.to_datetime(df_ativos['inicio_vig'], errors='coerce').dt.date
if not pd.api.types.is_datetime64_any_dtype(df_cancelamentos['data_cancelamento']):
    df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

# Se hoje for segunda-feira, usar intervalo sexta, sábado, domingo. Caso contrário, apenas ontem
if hasattr(yesterday, "weekday"):
    dia_semana = yesterday.weekday()
else:
    try:
        dia_semana = datetime.datetime.strptime(str(yesterday), "%Y-%m-%d").weekday()
    except Exception:
        dia_semana = None  # fallback

if today.weekday() == 0:  # Segunda-feira, pegar sexta, sábado e domingo
    sexta = yesterday - datetime.timedelta(days=2)
    sabado = yesterday - datetime.timedelta(days=1)
    domingo = yesterday
    dias_periodo = [sexta, sabado, domingo]

    # Filtro para o range de sexta a domingo
    df_ativos_periodo = df_ativos[df_ativos['inicio_vig'].isin(dias_periodo)]
    df_ativos_unidades = df_ativos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    df_cancelamentos_periodo = df_cancelamentos[df_cancelamentos['data_cancelamento'].isin(dias_periodo)]
    df_cancelamentos_unidades = df_cancelamentos_periodo.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    periodo_str = f'{sexta.strftime("%d/%m/%Y")} a {domingo.strftime("%d/%m/%Y")}'
    
    top_ativos_unidades = df_ativos_unidades.head(20)
    top_cancelamentos_unidades = df_cancelamentos_unidades.head(20)
else:
    # Apenas ontem
    df_ativos_ontem = df_ativos[df_ativos['inicio_vig'] == yesterday]
    df_ativos_ontem_unidades = df_ativos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    df_cancelamentos_ontem = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]
    df_cancelamentos_ontem_unidades = df_cancelamentos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

    periodo_str = yesterday.strftime('%d/%m/%Y')
    top_ativos_unidades = df_ativos_ontem_unidades.head(20)
    top_cancelamentos_unidades = df_cancelamentos_ontem_unidades.head(20)

# Criar figura com dois subplots lado a lado, ativos top 20 e cancelados top 20
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Gráfico 1: Ativos por Unidade (Top 20)
if not top_ativos_unidades.empty:
    top_ativos_unidades.plot(kind='barh', ax=ax1, color='#2ecc71', edgecolor='black', linewidth=0.5)
    ax1.set_title(f'Ativos por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')
    ax1.set_xticks([])
    ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax1.invert_yaxis()
    xmax1 = top_ativos_unidades.values.max() if len(top_ativos_unidades) > 0 else 1
    ax1.set_xlim(0, xmax1 + max(3, int(0.15 * xmax1)))
    for i, v in enumerate(top_ativos_unidades.values):
        ax1.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax1.transAxes, fontsize=12)
    ax1.set_title(f'Ativos por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')
    ax1.set_xticks([])


# Gráfico 2: Cancelados por Unidade (Top 20)
if not top_cancelamentos_unidades.empty:
    top_cancelamentos_unidades.plot(kind='barh', ax=ax2, color='#e74c3c', edgecolor='black', linewidth=0.5)
    ax2.set_title(f'Cancelados por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold', pad=15)
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')
    ax2.set_xticks([])
    ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax2.invert_yaxis()
    xmax2 = top_cancelamentos_unidades.values.max() if len(top_cancelamentos_unidades) > 0 else 1
    ax2.set_xlim(0, xmax2 + max(3, int(0.15 * xmax2)))
    for i, v in enumerate(top_cancelamentos_unidades.values):
        ax2.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title(f'Cancelados por Unidade (Top 20)\n{periodo_str}', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')
    ax2.set_xticks([])

plt.tight_layout()

# Salvar a imagem SOMENTE com o nome referente ao dia final do período (ontem/yesterday ou domingo se segunda)
output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
os.makedirs(output_path, exist_ok=True)
nome_arquivo = f'graficos_unidades_{yesterday.strftime("%d-%m")}.png'
image_path = os.path.join(output_path, nome_arquivo)
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Gráficos salvos em: {image_path}')

plt.show()

C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_51596\1957891330.py:55: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_51596\1957891330.py:76: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)


Gráficos salvos em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\graficos_unidades_09-02.png


C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_51596\1957891330.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### automação envio whatsapp

In [ ]:
# time.sleep(2)
# pyautogui.hotkey('win', 'e')
# time.sleep(4)
# pyautogui.hotkey('ctrl', 'l') 
# time.sleep(1.5)
# pyautogui.typewrite(r'C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(2.5)
# pyautogui.typewrite(r'graficos')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# #whatsapp - primeiro arquivo
# time.sleep(1.5)
# pyautogui.press('win')
# time.sleep(1.5)
# pyautogui.typewrite('whatsapp')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.typewrite("RELAT")
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')
# time.sleep(1.5)
######################################################whatsapp - segundo arquivo
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'a')
# time.sleep(2.5)
# pyautogui.typewrite(r'tabela')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# time.sleep(1.5)
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')


In [30]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()

In [ ]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()